# บทที่ 07 · ให้โมเดลเรียนรู้ โดยไม่แอบเห็นอนาคต

เรียนรู้ feature/label ตามเวลา train-only scaling, ridge classification, validation selection และการอ่านผล final test อย่างซื่อตรง

ก่อนเริ่ม: pandas และแนวคิด train/validation/test ติดตั้ง numpy==2.5.3 และ pandas==2.3.2 ใช้ Python 3.12 ไม่ต้องมี scikit-learn หรือ Webull SDK

ทุกข้อมูลเป็นสังเคราะห์ 420 session เลข 0–419 (index Python ไม่มีปฏิทิน/timezone จริง) ราคาเริ่ม 100 USD สมมติไม่มี overnight return, corporate actions หรือเงินปันผล ใช้ seed 7 และไม่เปลี่ยน seed หลังเห็นผล

## 1. สร้างโจทย์ที่รู้ที่มา

r_t = .25 × r_(t-1) + noise, noise มีส่วนเบี่ยงเบนมาตรฐาน .012 ความสัมพันธ์ .25 ถูกใส่ไว้โดยตั้งใจ จึงไม่ใช่หลักฐานว่าตลาดจริงมี pattern นี้

In [1]:
import platform
import numpy as np
import pandas as pd

print({"python": platform.python_version(), "numpy": np.__version__, "pandas": pd.__version__})
rng = np.random.default_rng(7)
returns = np.zeros(420)
noise = rng.normal(0, .012, len(returns))
for t in range(1, len(returns)):
    returns[t] = .25 * returns[t - 1] + noise[t]
closes = 100 * np.cumprod(1 + returns)
opens = np.r_[100, closes[:-1]]
assert np.all(returns > -1)
print({"sessions": len(returns), "seed": 7, "first_close": round(float(closes[0]), 6), "last_close": round(float(closes[-1]), 6)})


{'python': '3.12.14', 'numpy': '2.5.3', 'pandas': '2.3.2'}
{'sessions': 420, 'seed': 7, 'first_close': 100.0, 'last_close': 52.239133}


## 2. สร้างแถวที่พร้อมหลังปิด t

feature ใช้ผลตอบแทน t, t-1, t-2 ส่วน label เป็นทิศของเปิดถึงปิด t+1 เข้าซื้อได้เร็วสุดในสมมติฐานนี้ที่เปิด t+1 เหลือ 417 ตัวอย่างเพราะต้องมีอดีตครบและมี label

In [2]:
rows = []
for t in range(2, len(returns) - 1):
    rows.append((t, t + 1, returns[t], returns[t - 1], returns[t - 2], 1 if returns[t + 1] > 0 else -1, returns[t + 1]))
data = pd.DataFrame(rows, columns=["decision_session", "label_session", "r0", "r1", "r2", "y", "future_return"])
features = ["r0", "r1", "r2"]
X, y = data[features].to_numpy(), data.y.to_numpy()
print(data.head(5).round(6).to_string(index=False))


 decision_session  label_session        r0        r1        r2  y  future_return
                2              3 -0.002393  0.003585  0.000000 -1      -0.011285
                3              4 -0.011285 -0.002393  0.003585 -1      -0.008277
                4              5 -0.008277 -0.011285 -0.002393 -1      -0.013969
                5              6 -0.013969 -0.008277 -0.011285 -1      -0.002771
                6              7 -0.002771 -0.013969 -0.008277  1       0.015390


## 3. แยกชุดก่อนปรับสเกล

Train 240, validation 80, final test 95 ตัวอย่าง เว้น 1 decision row ที่ขอบเขตแต่ละจุดเพื่อให้ label ก่อนหน้าจบก่อน decision ของชุดถัดไป

ค่าเฉลี่ยและส่วนเบี่ยงเบนมาตรฐาน ddof=0 คำนวณเฉพาะ train แล้วใช้กับทุกชุด ส่วน intercept เป็นคอลัมน์หนึ่ง ไม่ลงโทษใน ridge

In [3]:
train_idx = np.arange(0, 240)
valid_idx = np.arange(241, 321)
test_idx = np.arange(322, len(data))
# One excluded decision row at each boundary; no overlapping outcome horizon.
assert data.label_session.iloc[train_idx[-1]] < data.decision_session.iloc[valid_idx[0]]
assert data.label_session.iloc[valid_idx[-1]] < data.decision_session.iloc[test_idx[0]]
mu = X[train_idx].mean(axis=0)
sigma = X[train_idx].std(axis=0, ddof=0)
sigma = np.where(sigma == 0, 1.0, sigma)
Z = np.column_stack([np.ones(len(X)), (X - mu) / sigma])
print(pd.DataFrame({"split": ["train", "validation", "test"], "rows": [len(train_idx), len(valid_idx), len(test_idx)], "first_decision": [data.decision_session.iloc[i[0]] for i in [train_idx, valid_idx, test_idx]], "last_label": [data.label_session.iloc[i[-1]] for i in [train_idx, valid_idx, test_idx]]}).to_string(index=False))
print("Training means:", np.round(mu, 8))


     split  rows  first_decision  last_label
     train   240               2         242
validation    80             243         323
      test    95             324         419
Training means: [-0.00238319 -0.00239514 -0.00235402]


## 4. เลือก alpha บน validation

แก้สมการ ridge ด้วย np.linalg.solve โดย label เป็น -1/+1 แล้วใช้คะแนน > 0 เป็นคำทำนายขึ้น คะแนนไม่ใช่ probability

ตัวเลือก .1, 1, 10, 100 ถูกกำหนดไว้ก่อน เลือก validation balanced accuracy สูงสุด ถ้าเสมอเลือก alpha มากกว่า ได้ alpha 1 แล้วตรึงทุกอย่าง ไม่มี refit และไม่มีเลือก threshold จาก test

In [4]:
def fit_ridge(design, labels, alpha):
    penalty = np.eye(design.shape[1]) * alpha
    penalty[0, 0] = 0
    return np.linalg.solve(design.T @ design + penalty, design.T @ labels)


def classification_metrics(actual, predicted):
    recalls = [np.mean(predicted[actual == label] == label) for label in [-1, 1]]
    return {"accuracy": float(np.mean(actual == predicted)), "balanced_accuracy": float(np.mean(recalls))}


alphas = [.1, 1.0, 10.0, 100.0]
models, validation_rows = {}, []
for alpha in alphas:
    models[alpha] = fit_ridge(Z[train_idx], y[train_idx], alpha)
    prediction = np.where(Z[valid_idx] @ models[alpha] > 0, 1, -1)
    validation_rows.append({"alpha": alpha, **classification_metrics(y[valid_idx], prediction)})
# Predetermined tie-break: choose the stronger regularization.
chosen = max(validation_rows, key=lambda row: (row["balanced_accuracy"], row["alpha"]))["alpha"]
frozen_coef = models[chosen].copy()
print(pd.DataFrame(validation_rows).round(6).to_string(index=False))
print({"selected_alpha": chosen, "selection_metric": "validation balanced accuracy", "refit": False})


 alpha  accuracy  balanced_accuracy
   0.1     0.550              0.550
   1.0     0.550              0.550
  10.0     0.525              0.525
 100.0     0.500              0.500
{'selected_alpha': 1.0, 'selection_metric': 'validation balanced accuracy', 'refit': False}


## 5. เปิด final test เพียงเพื่อรายงาน

Ridge ทายถูก 52/95 = 54.7368% ขณะที่ persistence 55.7895% จึงยังไม่เหนือ baseline ด้านความแม่น ดู confusion matrix ด้วย: ทายลงถูก 43 แต่ทายขึ้นถูกเพียง 9 จาก 48 แถวที่ขึ้นจริง

In [5]:
test_prediction = np.where(Z[test_idx] @ frozen_coef > 0, 1, -1)
majority_label = 1 if (y[train_idx] == 1).sum() > (y[train_idx] == -1).sum() else -1
majority_prediction = np.full(len(test_idx), majority_label)
persistence_prediction = np.where(X[test_idx, 0] > 0, 1, -1)
predictions = {"ridge": test_prediction, "training_majority": majority_prediction, "persistence": persistence_prediction}
scores = pd.DataFrame({name: classification_metrics(y[test_idx], pred) for name, pred in predictions.items()}).T
print(scores.round(6).to_string())
matrix = pd.crosstab(pd.Series(y[test_idx], name="actual"), pd.Series(test_prediction, name="predicted")).reindex(index=[-1, 1], columns=[-1, 1], fill_value=0)
print(matrix.to_string())


                   accuracy  balanced_accuracy
ridge              0.547368           0.551197
training_majority  0.494737           0.500000
persistence        0.557895           0.557846
predicted  -1   1
actual           
-1         43   4
 1         39   9


## 6. แยกคุณภาพคำทำนายออกจากกำไร

แปลงขึ้นเป็น Long ลงเป็น Cash ใช้หุ้นเศษส่วน ไม่ชอร์ต/leverage เงินสดไม่มีดอกเบี้ย หักต้นทุน .1% ของ NAV ก่อนเปลี่ยนสถานะทุกครั้งตามบท 5 เริ่มเงินสดและปิดสถานะท้ายช่วง

ผลตอบแทนสุทธิ +4.1508% เป็นเส้นทางสังเคราะห์เดียว ไม่ยืนยันข้อได้เปรียบจริง Accuracy ให้น้ำหนักทุกแถวเท่ากัน แต่ P&L สนใจขนาดผลตอบแทนและช่วงที่มี exposure

In [6]:
def strategy_result(future_returns, long_target, cost=.001):
    equity, previous, switches = 1.0, 0, 0
    path = [equity]
    for ret, target in zip(future_returns, long_target):
        change = abs(int(target) - previous)
        equity *= (1 - cost * change) * (1 + int(target) * ret)
        switches += change
        path.append(equity)
        previous = int(target)
    equity *= 1 - cost * previous
    path[-1] = equity
    switches += previous
    values = np.array(path)
    return {"return_pct": (equity - 1) * 100, "max_drawdown_pct": np.min(values / np.maximum.accumulate(values) - 1) * 100, "long_exposure_pct": np.mean(long_target) * 100, "cost_events": switches}


test_returns = data.future_return.iloc[test_idx].to_numpy()
trading = pd.DataFrame({
    "ridge_long_cash_net": strategy_result(test_returns, test_prediction == 1),
    "always_long_net": strategy_result(test_returns, np.ones(len(test_idx), dtype=int)),
    "cash_no_interest": strategy_result(test_returns, np.zeros(len(test_idx), dtype=int)),
}).T
print(trading.round(6).to_string())


                     return_pct  max_drawdown_pct  long_exposure_pct  cost_events
ridge_long_cash_net    4.150785         -2.224978          13.684211         20.0
always_long_net       -0.247006        -13.834644         100.000000          2.0
cash_no_interest       0.000000          0.000000           0.000000          0.0


## 7. ตรวจการไหลของข้อมูล

การเปลี่ยน feature ฝั่ง test ต้องไม่เปลี่ยนค่าเฉลี่ยหรือโมเดลที่ fit บน train ตรวจ label กับราคาเปิด/ปิดจริงในข้อมูลสังเคราะห์ทุกแถว ตรวจขอบเขต train/validation/test และต้นทุนเข้าออกด้วยตัวอย่างเล็ก

In [7]:
assert len(data) == 417 and len(test_idx) == 95
assert np.allclose(Z[train_idx, 1:].mean(axis=0), 0, atol=1e-12)
# Altering future/test data must not change training parameters or selected alpha.
tampered_X = X.copy()
tampered_X[test_idx] += 1000
assert np.array_equal(tampered_X[train_idx], X[train_idx])
assert np.array_equal(tampered_X[train_idx].mean(axis=0), mu)
assert np.array_equal(fit_ridge(Z[train_idx], y[train_idx], chosen), frozen_coef)
# Rebuild a known prefix from a series whose distant future has been altered.
altered_returns = returns.copy()
altered_returns[325:] = .9
for t in range(2, 243):
    past_features = np.array([altered_returns[t], altered_returns[t - 1], altered_returns[t - 2]])
    assert np.array_equal(past_features, X[t - 2])
assert matrix.to_numpy().sum() == 95
# Every label exactly matches the next session's executable synthetic open-close return.
for row in data.itertuples():
    assert np.isclose(row.future_return, closes[row.label_session] / opens[row.label_session] - 1)
    assert row.r0 == returns[row.decision_session]
assert strategy_result(np.array([.01, -.02]), np.array([1, 0]))["cost_events"] == 2
assert np.isclose(strategy_result(np.array([.01, -.02]), np.array([1, 0]))["return_pct"], ((.999**2) * 1.01 - 1) * 100)
print("PASS: time boundaries, train-only scaling, next-session labels, confusion matrix, and costs")


PASS: time boundaries, train-only scaling, next-session labels, confusion matrix, and costs


## แบบฝึกหัดและเฉลย

เพิ่ม 1000 เฉพาะ feature ของ test แล้ว fit ค่าเฉลี่ย train ใหม่: ค่าเฉลี่ยต้องไม่เปลี่ยน แต่ prediction บน input ใหม่เปลี่ยนได้ เพราะ input เปลี่ยน ไม่ใช่โมเดลเรียนเพิ่ม

เปลี่ยน label ให้ครอบคลุม 5 session ถัดไป: อย่าคง gap=1 โดยอัตโนมัติ ต้องตรวจช่วงผลลัพธ์ที่ทับขอบเขตแล้วกันแถวที่ยังมี outcome ในช่วงประเมิน

อย่าเปลี่ยน alpha หรือ threshold ให้ test ดีขึ้นแล้วยังเรียกว่า test อิสระ บันทึกการทดลองเดิมและใช้ข้อมูลใหม่ที่ยังไม่เห็นสำหรับประเมินรุ่นถัดไป

## แหล่งอ้างอิงและสถานะการรัน

Yves Hilpisch, Python for Algorithmic Trading; https://github.com/yhilpisch/py4at; บท 5 หน้า 132, 134, 146 (PDF 152, 154, 166)

- https://scikit-learn.org/stable/common_pitfalls.html#data-leakage
- https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.TimeSeriesSplit.html
- https://numpy.org/doc/stable/reference/generated/numpy.linalg.solve.html

ตรวจแหล่งออนไลน์ 11 กันยายน 2026 บทเรียน ข้อมูล และโค้ดเขียนใหม่ ไม่ได้แนบหนังสือหรือแจกโค้ดต้นฉบับของ Hilpisch

มีผลจากการรัน Python ทุกเซลล์ตามลำดับจาก state ว่างแนบไว้แล้ว ไม่มีการเชื่อมเครือข่าย กด Restart Kernel แล้ว Run All เพื่อทำซ้ำใน Jupyter ได้